# 16 - Fine-tuned 2.5D attention-MIL model on LLM soft labels

Follows the recipe of the strongest public notebooks, re-implemented here:

* **labels:** notebook 11c's soft report labels (probabilities, not 0/1);
* **input:** three neighbouring slices stacked as the RGB channels of one image (2.5D),
  `K` such windows per plane from notebook 13's cache, all three planes;
* **model:** an ImageNet-pretrained timm backbone, **fine-tuned**, one feature vector per
  window plus a learned plane embedding, then an attention pooling with separate
  weights for each finding (a torn ACL shows on two slices, OA across many);
* **augmentation (training only):** random window positions plus medprep `augment`
  (small in-plane rotation and shift, intensity scale/shift; never flips, because
  right knees are already mirrored to the left-knee convention).

**Leakage control.** The 58 expert-labelled studies, and every study that shares a
report with one of them, are never trained on and never used for any choice. Epochs
are selected on fold 0 of notebook 12 (weakly labelled studies only, soft-label BCE);
the expert-labelled studies are scored once, at the end.

**Settings:** GPU T4 x2, internet ON (timm weights, medprep). Run `SMOKE = True` first:
it trains 1 epoch on 64 studies and prints the projected full-run time.
**Inputs:** notebook 11 (`weak_labels_v1.csv`), 11c (`soft_labels_v1.csv`),
12 (`folds_v1.csv`), 13 (cache + manifest).
**Outputs:** `model/` (weights + config), `gold_eval_v1.json`, `gold_preds_v1.csv`,
`history_v1.csv`.

In [ ]:
%env GIT_TERMINAL_PROMPT=0
%pip install -q "medprep-starter @ git+https://github.com/Z-bros/Med_Im_Prepper.git"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# The competition mount path has differed between Kaggle runtimes; take the first that has train.csv.
_SLUG = "rsna-knee-abnormality-detection"
_roots = [Path("/kaggle/input/competitions") / _SLUG, Path("/kaggle/input") / _SLUG,
          *sorted(Path("/kaggle/input").glob("*/" + _SLUG)), *sorted(Path("/kaggle/input").glob("*"))]
ROOT = next((r for r in _roots if (r / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("competition train.csv not found; /kaggle/input has: "
                            + str(sorted(str(p) for p in Path("/kaggle/input").glob("*/*"))[:50]))
print("competition data:", ROOT)
OUT = Path("/kaggle/working")
TARGET_COLUMNS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 42
from pathlib import Path


def find_input(name):
    """Locate one attached upstream file by name.

    Only notebook outputs and datasets are searched: walking the competition
    mount would visit millions of DICOM files.
    """
    hits = []
    for base in (Path("/kaggle/input/notebooks"), Path("/kaggle/input/datasets"),
                 Path("/kaggle/input")):
        if base.is_dir():
            for child in base.iterdir():
                if (child.is_dir()
                        and child.name not in ("competitions", "notebooks", "datasets")
                        and not (child / "train_series").exists()
                        and not (child / "test_series").exists()):
                    hits += list(child.rglob(name))
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly one attached '{name}', found {len(hits)}: {hits[:5]}. "
            "Attach the upstream notebook output via Add Input."
        )
    return hits[0]

import copy
import json
import random
import time

from medprep import AugmentConfig, augment
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SMOKE = False
LABELS = "soft"      # "soft": 11c soft labels (distillation) | "regex": notebook 11 weak scores (ablation)
ARCH = "convnext_tiny.fb_in22k_ft_in1k"
K_TRAIN = 8          # windows per plane, random positions
K_EVAL = 8           # windows per plane, evenly spaced
EPOCHS = 8
BATCH = 4            # studies per step
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY = 1e-4, 1e-3, 0.05
VAL_FOLD = 0
TOP_K_AVG = 3
AUG = AugmentConfig(rotation_degrees=8.0, rotation_axes=(1, 2), rotation_probability=0.5,
                    translation_pixels=(0.0, 12.0, 12.0), translation_probability=0.5,
                    intensity_scale_range=(0.9, 1.1), intensity_shift_range=(-0.05, 0.05),
                    intensity_probability=0.5)
if SMOKE:
    EPOCHS = 1


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
# Shared 2.5D attention-MIL model: identical in notebooks 16 and 15.
import numpy as np
import timm
import torch
from torch import nn

MIL_VERSION = "mil-v1-2p5d-3slice-perfinding-attn-planeemb"
N_PLANES, CACHE_SLICES = 3, 16
WINDOW_CENTRES = np.arange(1, CACHE_SLICES - 1)  # a window is slices c-1, c, c+1


def eval_centres(k):
    """k window centres spread evenly over the volume (deterministic)."""
    return WINDOW_CENTRES[np.round(np.linspace(0, len(WINDOW_CENTRES) - 1, k)).astype(int)]


def windows_from_volume(volume, centres_per_plane):
    """volume: (3, 16, H, W) uint8 -> (3, K, 3, H, W) float32 in [0, 1]."""
    out = np.stack([np.stack([volume[p, c - 1:c + 2] for c in centres_per_plane[p]])
                    for p in range(N_PLANES)])
    return out.astype(np.float32) / 255.0


class KneeMIL(nn.Module):
    """Backbone per 3-slice window + plane embedding, then one attention pooling
    per finding over all windows of the study (missing planes masked out)."""

    def __init__(self, arch, n_out=12, pretrained=True, drop=0.2):
        super().__init__()
        self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0)
        f = self.backbone.num_features
        self.plane_emb = nn.Parameter(torch.zeros(N_PLANES, f))
        self.norm = nn.LayerNorm(f)
        self.att = nn.Sequential(nn.Linear(f, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n_out))
        self.cls_w = nn.Parameter(torch.randn(n_out, f) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, present):
        """x: (B, P, K, 3, H, W) in [0, 1]; present: (B, P) -> logits (B, n_out)."""
        b, p, k = x.shape[:3]
        feats = self.backbone((x.flatten(0, 2) - self.mean) / self.std)
        h = feats.view(b, p, k, -1).float() + self.plane_emb[None, :, None]
        h = self.norm(h.flatten(1, 2))                                  # (B, P*K, F)
        a = self.att(h).float()                                         # (B, P*K, n)
        keep = present[:, :, None].expand(b, p, k).reshape(b, p * k)
        a = torch.softmax(a.masked_fill(keep[..., None] <= 0, -1e4), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

In [ ]:
# Data-free check: shapes, finite gradients, missing plane gets no attention weight.
_m = KneeMIL("resnet18", pretrained=False).train()
_x = torch.rand(2, N_PLANES, 2, 3, 32, 32)
_present = torch.tensor([[1.0, 1.0, 1.0], [1.0, 0.0, 1.0]])
_out = _m(_x, _present)
assert _out.shape == (2, 12) and torch.isfinite(_out).all()
_out.sum().backward()
assert all(torch.isfinite(q.grad).all() for q in _m.parameters() if q.grad is not None)
_x2 = _x.clone(); _x2[1, 1] = torch.rand_like(_x2[1, 1])  # change only the missing plane
_m.eval()
with torch.no_grad():
    assert torch.allclose(_m(_x, _present)[1], _m(_x2, _present)[1], atol=1e-5)
assert windows_from_volume(np.zeros((3, 16, 8, 8), np.uint8), [eval_centres(4)] * 3).shape == (3, 4, 3, 8, 8)
print("MIL self-test passed")

## 1. Inputs and the train / epoch-selection / expert split

In [ ]:
seed_everything(SEED)
weak = pd.read_csv(find_input("weak_labels_v1.csv"), dtype={"StudyInstanceUID": str})
try:
    if LABELS == "regex":
        raise FileNotFoundError  # ablation: train on notebook 11's regex scores, same everything else
    soft = pd.read_csv(find_input("soft_labels_v1.csv"), dtype={"StudyInstanceUID": str})
    LABEL_SOURCE = "11c soft labels"
except FileNotFoundError:
    if LABELS == "regex":
        soft = weak[["StudyInstanceUID"]].assign(**{t: weak[t + "__weak"] for t in TARGET_COLUMNS})
        LABEL_SOURCE = "11 regex weak scores"
    elif not SMOKE:
        raise
    else:  # smoke run only: wiring check with notebook 11's regex scores while 11c is still running
        soft = weak[["StudyInstanceUID"]].assign(**{t: weak[t + "__weak"] for t in TARGET_COLUMNS})
        LABEL_SOURCE = "regex weak scores (smoke fallback)"
print("training labels:", LABEL_SOURCE)
folds = pd.read_csv(find_input("folds_v1.csv"), dtype={"StudyInstanceUID": str})
manifest_path = find_input("preprocess_manifest_v1.csv")
manifest = pd.read_csv(manifest_path, dtype={"StudyInstanceUID": str}).drop_duplicates(
    "StudyInstanceUID", keep="last")
CACHE = manifest_path.parent / "cache"
PREPROC_VERSION = manifest["version"].iloc[0]
assert manifest["version"].eq(PREPROC_VERSION).all()

data = (folds[["StudyInstanceUID", "fold", "group", "is_gold"]]
        .merge(soft[["StudyInstanceUID"] + TARGET_COLUMNS], on="StudyInstanceUID", validate="one_to_one")
        .merge(weak[["StudyInstanceUID"] + [t + "__gold" for t in TARGET_COLUMNS]],
               on="StudyInstanceUID", validate="one_to_one")
        .merge(manifest[["StudyInstanceUID", "status"]], on="StudyInstanceUID", how="left"))
assert data["status"].eq("ok").all(), "every study needs a usable cache"
gold_groups = set(data.loc[data["is_gold"].eq(1), "group"])
near_gold = data["group"].isin(gold_groups) & data["is_gold"].eq(0)
is_gold = data["is_gold"].eq(1)
train_df = data[~is_gold & ~near_gold & data["fold"].ne(VAL_FOLD)]
val_df = data[~is_gold & ~near_gold & data["fold"].eq(VAL_FOLD)]
gold_df = data[is_gold]
print("excluded (share a report with an expert-labelled study):", int(near_gold.sum()))
N_TRAIN_FULL = len(train_df)
if SMOKE:
    train_df, val_df = train_df.sample(64, random_state=SEED), val_df.sample(32, random_state=SEED)
print("train", len(train_df), "| epoch selection", len(val_df), "| expert-labelled (final only)", len(gold_df))
assert not set(train_df["StudyInstanceUID"]) & set(gold_df["StudyInstanceUID"])

## 2. Data

In [ ]:
class StudyWindows(Dataset):
    def __init__(self, frame, train):
        self.ids = frame["StudyInstanceUID"].tolist()
        self.y = frame[TARGET_COLUMNS].to_numpy(np.float32)
        self.train = train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        with np.load(CACHE / f"{self.ids[i]}.npz") as z:
            vol, present = z["volume"], z["present"]
        if self.train:
            centres = [np.sort(np.random.choice(WINDOW_CENTRES, K_TRAIN, replace=False))
                       for _ in range(N_PLANES)]
        else:
            centres = [eval_centres(K_EVAL)] * N_PLANES
        x = windows_from_volume(vol, centres)
        if self.train:
            for p in range(N_PLANES):
                if present[p]:
                    for k in range(x.shape[1]):
                        x[p, k] = np.clip(augment(x[p, k], config=AUG,
                                                  seed=np.random.randint(2**31))["image"], 0, 1)
        return torch.from_numpy(x), torch.from_numpy(present.astype(np.float32)), torch.from_numpy(self.y[i])


def worker_init(worker_id):
    np.random.seed((torch.initial_seed() + worker_id) % 2**32)


def loader(frame, train):
    return DataLoader(StudyWindows(frame, train), batch_size=BATCH if train else 2 * BATCH,
                      shuffle=train, num_workers=4, drop_last=train, pin_memory=True,
                      worker_init_fn=worker_init, persistent_workers=False)


train_dl, val_dl = loader(train_df, True), loader(val_df, False)
_x, _p, _y = next(iter(train_dl))
print("batch:", tuple(_x.shape), tuple(_p.shape), tuple(_y.shape), "| range", float(_x.min()), float(_x.max()))

## 3. Model and optimiser

In [ ]:
device = torch.device("cuda")
model = KneeMIL(ARCH, pretrained=True)
prior = train_df[TARGET_COLUMNS].mean().clip(1e-3, 1 - 1e-3).to_numpy()
with torch.no_grad():
    model.cls_b.copy_(torch.from_numpy(np.log(prior / (1 - prior))).float())
if hasattr(model.backbone, "set_grad_checkpointing"):
    model.backbone.set_grad_checkpointing(True)
model = model.to(device)
net = nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
print("GPUs:", torch.cuda.device_count(), "| backbone params (M):",
      round(sum(p.numel() for p in model.backbone.parameters()) / 1e6, 1))

bb = list(model.backbone.parameters())
bb_ids = {id(p) for p in bb}
head = [p for p in model.parameters() if id(p) not in bb_ids]
opt = torch.optim.AdamW([{"params": bb, "lr": LR_BACKBONE}, {"params": head, "lr": LR_HEAD}],
                        weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD],
                                            total_steps=EPOCHS * len(train_dl), pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")
bce = nn.functional.binary_cross_entropy_with_logits


@torch.no_grad()
def predict(dl):
    net.eval()
    out = []
    for x, p, _ in tqdm(dl, desc="predict", leave=False, mininterval=60):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.sigmoid(net(x.to(device, non_blocking=True), p.to(device)).float()).cpu())
    return torch.cat(out).numpy()


def soft_bce(pred, frame):
    y = torch.from_numpy(frame[TARGET_COLUMNS].to_numpy(np.float32))
    p = torch.from_numpy(pred).clamp(1e-6, 1 - 1e-6)
    return float(nn.functional.binary_cross_entropy(p, y))

## 4. Train

In [ ]:
history, top = [], []  # top: (val_bce, epoch, cpu state_dict)
t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    net.train()
    t0, losses = time.time(), []
    for x, p, y in tqdm(train_dl, desc=f"epoch {epoch}/{EPOCHS}", mininterval=60):
        x, p, y = x.to(device, non_blocking=True), p.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = bce(net(x, p).float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        losses.append(loss.item())
    train_secs = time.time() - t0
    val_bce = soft_bce(predict(val_dl), val_df)
    history.append({"epoch": epoch, "train_bce": float(np.mean(losses)), "val_bce": val_bce,
                    "train_seconds": round(train_secs), "total_seconds": round(time.time() - t_start)})
    print(history[-1])
    top.append((val_bce, epoch, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}))
    top = sorted(top, key=lambda t: t[0])[:TOP_K_AVG]
    torch.save(top[0][2], OUT / "best_epoch_state.pt")  # survives a session timeout

if SMOKE:
    per_study = history[0]["train_seconds"] / len(train_df)
    print(f"SMOKE: {per_study:.2f} s per training study per epoch")
    print(f"projected full run: {per_study * N_TRAIN_FULL * 8 / 3600:.1f} h for 8 epochs on "
          f"{N_TRAIN_FULL} studies, plus about 10% for validation")

## 5. Weight averaging of the best epochs (chosen on fold 0, never on expert labels)

In [ ]:
avg = {k: (sum(t[2][k].float() for t in top) / len(top)).to(top[0][2][k].dtype)
       if top[0][2][k].is_floating_point() else top[0][2][k] for k in top[0][2]}
candidates = {"best_epoch": top[0][2], f"avg_top{len(top)}": avg}
scores = {}
for name, state in candidates.items():
    model.load_state_dict(state)
    scores[name] = soft_bce(predict(val_dl), val_df)
CHOSEN = min(scores, key=scores.get)
model.load_state_dict(candidates[CHOSEN])
print("fold-0 soft BCE:", {k: round(v, 4) for k, v in scores.items()}, "-> using", CHOSEN,
      "| epochs", [t[1] for t in top])

## 6. Expert-labelled evaluation (once)

In [ ]:
from sklearn.metrics import roc_auc_score


def safe_auc(y, p):
    """AUC, or NaN when only one class is present (the metric is undefined)."""
    y = np.asarray(y)
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan


def auc_table(gold, scores, n_boot=2000, seed=0):
    """Per-target AUC with a percentile bootstrap interval over studies."""
    rng = np.random.default_rng(seed)
    rows = []
    for t in TARGET_COLUMNS:
        y, p = gold[t].to_numpy(), scores[t].to_numpy()
        boots = []
        for _ in range(n_boot):
            i = rng.integers(0, len(y), len(y))
            a = safe_auc(y[i], p[i])
            if np.isfinite(a):
                boots.append(a)
        rows.append({"target": t, "auc": safe_auc(y, p),
                     "ci_low": np.percentile(boots, 2.5) if boots else np.nan,
                     "ci_high": np.percentile(boots, 97.5) if boots else np.nan,
                     "n_pos": int(y.sum()), "n_neg": int((1 - y).sum())})
    return pd.DataFrame(rows)


def macro_auc(table):
    """Challenge metric: mean of per-target AUCs; undefined targets excluded and counted."""
    return float(table["auc"].mean(skipna=True)), int(table["auc"].notna().sum())

gold_pred = pd.DataFrame(predict(loader(gold_df, False)), columns=TARGET_COLUMNS, index=gold_df.index)
gold_y = gold_df[[t + "__gold" for t in TARGET_COLUMNS]].set_axis(TARGET_COLUMNS, axis=1).astype(int)
label_ref = gold_df[TARGET_COLUMNS]  # the soft labels themselves, for reference only
table = auc_table(gold_y, gold_pred)
table["soft_label_auc"] = auc_table(gold_y, label_ref)["auc"]
display(table.round(3))
macro, n_eval = macro_auc(table)
print(f"expert-labelled macro AUC (fine-tuned MIL): {macro:.4f} over {n_eval}/12 targets")
print("reference: notebook 14 frozen ResNet-18 OOF 0.6629; soft labels themselves "
      f"{macro_auc(table.assign(auc=table['soft_label_auc']))[0]:.4f} (not available at test time)")
if SMOKE:
    print("SMOKE run: this number is from 64 training studies and 1 epoch; it is a wiring check only.")

## 7. Save

In [ ]:
MODEL = OUT / "model"
MODEL.mkdir(exist_ok=True)
torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in model.state_dict().items()},
           MODEL / "mil_weights.pt")
config = {
    "model_kind": "mil", "mil_version": MIL_VERSION, "arch": ARCH, "k_eval": K_EVAL,
    "preproc_version": PREPROC_VERSION, "target_columns": TARGET_COLUMNS, "seed": SEED,
    "smoke": SMOKE, "label_source": LABEL_SOURCE, "epochs": EPOCHS, "batch": BATCH, "lr_backbone": LR_BACKBONE,
    "lr_head": LR_HEAD, "weight_decay": WEIGHT_DECAY, "chosen": CHOSEN,
    "top_epochs": [t[1] for t in top], "val_fold": VAL_FOLD,
    "n_train": len(train_df), "n_val": len(val_df),
    "fallback_prevalence": {t: float(v) for t, v in zip(TARGET_COLUMNS, prior)},
    "versions": {"torch": torch.__version__, "timm": timm.__version__},
}
(MODEL / "config.json").write_text(json.dumps(config, indent=2))
pd.DataFrame(history).to_csv(OUT / "history_v1.csv", index=False)
pd.concat([gold_df[["StudyInstanceUID"]], gold_pred.add_suffix("__pred")], axis=1).to_csv(
    OUT / "gold_preds_v1.csv", index=False)
(OUT / "gold_eval_v1.json").write_text(json.dumps({
    "macro_auc": macro, "evaluable_targets": n_eval, "smoke": SMOKE,
    "per_target": table.to_dict("records")}, indent=2, default=float))
print("saved", MODEL)